In [1]:
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
ARTICLES_COLLECTION_NAME = os.getenv("MONGO_ARTICLES_NAME")
NODES_COLLECTION_NAME = os.getenv("MONGO_NODES_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, ARTICLES_COLLECTION_NAME, NODES_COLLECTION_NAME]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()

# Create MongoDB client with TLS verification
try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]

    # Define collections
    articles_collection = db[ARTICLES_COLLECTION_NAME]  # Stores articles
    nodes_collection = db[NODES_COLLECTION_NAME]  # Stores extracted nodes

    # Verify connection
    client.admin.command('ping')
    print(f"✅ Connected to MongoDB Atlas! Database: {DB_NAME}")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    exit()

✅ Connected to MongoDB Atlas! Database: tuone


In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
openAI_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openAI_api_key)

def ping_openai(client):
    try:
        response = client.models.list()
        print("✅ Successfully connected to OpenAI API!")
        print(f"Available Models: {[model.id for model in response.data]}")
    except Exception as e:
        print(f"❌ OpenAI API Connection Error: {e}")
ping_openai(client)

✅ Successfully connected to OpenAI API!
Available Models: ['gpt-4o-realtime-preview-2024-12-17', 'gpt-4o-audio-preview-2024-12-17', 'dall-e-3', 'dall-e-2', 'gpt-4o-audio-preview-2024-10-01', 'gpt-4o-realtime-preview-2024-10-01', 'gpt-4o-transcribe', 'gpt-4o-mini-transcribe', 'gpt-4o-realtime-preview', 'babbage-002', 'gpt-4o-mini-tts', 'tts-1-hd-1106', 'text-embedding-3-large', 'gpt-4', 'text-embedding-ada-002', 'omni-moderation-latest', 'computer-use-preview', 'computer-use-preview-2025-03-11', 'tts-1-hd', 'gpt-4o-mini-audio-preview', 'gpt-4o-audio-preview', 'o1-preview-2024-09-12', 'gpt-4o-mini-realtime-preview', 'gpt-4o-mini-realtime-preview-2024-12-17', 'gpt-3.5-turbo-instruct-0914', 'gpt-4o-mini-search-preview', 'tts-1-1106', 'davinci-002', 'gpt-3.5-turbo-1106', 'gpt-4-turbo', 'gpt-4-0125-preview', 'gpt-3.5-turbo-instruct', 'gpt-3.5-turbo', 'gpt-4-turbo-preview', 'chatgpt-4o-latest', 'gpt-4o-mini-search-preview-2025-03-11', 'gpt-4o-2024-11-20', 'whisper-1', 'gpt-3.5-turbo-0125', 'g

In [3]:
def combine_paragraphs(article):
    paragraphs = article.get('paragraphs', [])
    # Handle missing or empty paragraphs
    if not paragraphs:
        print("⚠️ No paragraphs found in the article.")
        return ""

    combined_text = ""
    for para_obj in paragraphs:
        for key in sorted(para_obj.keys()):
            combined_text += para_obj[key].strip() + " "

    return combined_text.strip()

def read_prompt_from_file_only(file_path):
    with open(file_path, 'r') as file:
        prompt = file.read()
    return prompt

In [4]:
# this function reconstructs the nodes into the format they were originally outputted

import json
from collections import defaultdict

def reconstruct_function_call_arguments(validated_nodes):
    grouped_nodes = defaultdict(list)

    for node in validated_nodes:
        node_type = node["type"]
        reconstructed_node = {
            "id": node.get("id"),
            "type": node_type,
            "name": node.get("name"),
        }

        # Include optional fields if present
        if node.get("location"):
            reconstructed_node["location"] = {
                "city": node["location"].get("city", ""),
                "country": node["location"].get("country", "")
            }

        if node.get("amount") is not None:
            reconstructed_node["amount"] = node["amount"]
        if node.get("status") is not None:
            reconstructed_node["status"] = node["status"]

        grouped_nodes[node_type].append(reconstructed_node)

    return json.dumps({"nodes": grouped_nodes}, ensure_ascii=False)

In [6]:
offset = 0
limit = 50
output_file = "finetuning_nodes.jsonl"

PROMPT_PATH = "/Users/ben/Documents/bruegel/data_new/WORKING/INDUSTRY/tuone_v6/Entity-relationships/prompts/entities-only.txt"
system_prompt = read_prompt_from_file_only(PROMPT_PATH)

articles_to_process = list(
    articles_collection.find(
        {"meta.ID": {"$regex": "^27"}},  # <-- Add this filter
        {"_id": 1, "paragraphs": 1, "nodes_ben": 1, "validation": 1}       # Keep projection as is
    )
    .sort("_id", -1)  # Sort by MongoDB ObjectId (descending)
    .skip(offset)    # Skip first `offset` articles
    .limit(limit)    # Limit the number of articles
)

print(f"📦 Exporting up to {limit} validated examples starting from offset {offset}...")


with open(output_file, "w", encoding="utf-8") as f_out:
    for article in articles_to_process:
        articleID = str(article["_id"])

        if article.get("validation") is not True:
            print(f"⏭️ Skipping Article ID: {articleID} - not validated")
            continue

        text = combine_paragraphs(article)
        user_content = f"Here is the article: {text}"

        nodes_ben = article.get("nodes_ben")
        if not nodes_ben:
            print(f"⚠️ Article ID: {articleID} has no nodes_ben — skipping")
            continue

        assistant_content = reconstruct_function_call_arguments(nodes_ben)

        # Create fine-tuning format
        messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_content
            },
            {
                "role": "assistant",
                "content": assistant_content
            }
        ]

        f_out.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
        print(f"✅ Wrote example for Article ID: {articleID}")

print(f"\n🎉 Finished exporting fine-tuning examples to {output_file}")


📦 Exporting up to 50 validated examples starting from offset 0...
✅ Wrote example for Article ID: 67b20c230554959fd7264b95
✅ Wrote example for Article ID: 67b20c220554959fd7264b94
✅ Wrote example for Article ID: 67b20c220554959fd7264b93
✅ Wrote example for Article ID: 67b20c210554959fd7264b92
⏭️ Skipping Article ID: 67b20c210554959fd7264b91 - not validated
✅ Wrote example for Article ID: 67b20c210554959fd7264b90
✅ Wrote example for Article ID: 67b20c200554959fd7264b8f
✅ Wrote example for Article ID: 67b20c200554959fd7264b8e
✅ Wrote example for Article ID: 67b20c1f0554959fd7264b8d
✅ Wrote example for Article ID: 67b20c1f0554959fd7264b8c
✅ Wrote example for Article ID: 67b20c1f0554959fd7264b8b
✅ Wrote example for Article ID: 67b20c1e0554959fd7264b8a
✅ Wrote example for Article ID: 67b20c1e0554959fd7264b89
✅ Wrote example for Article ID: 67b20c1d0554959fd7264b88
✅ Wrote example for Article ID: 67b20c1d0554959fd7264b87
✅ Wrote example for Article ID: 67b20c1d0554959fd7264b86
✅ Wrote exampl